# S4_3 — Leaf-JEPA Main Pretraining Loop

**Leaf-JEPA IRP** | Stage 4 — Leaf JEPA Pretraining


The core Stage 4 notebook. Runs the full domain pretraining for PT_EPOCHS epochs.

**Expected runtime:**
- ViT-H/14, batch=128, A100 (80GB): ~8–12 min/epoch → 100 epochs ≈ 14–20 hours
- ViT-H/14, batch=64,  V100 (32GB): ~15–20 min/epoch → 100 epochs ≈ 25–33 hours
- ViT-B/16, batch=256, A100 (80GB): ~3–5  min/epoch  → 100 epochs ≈ 5–8 hours

**Resuming:** If training is interrupted, set RESUME_FROM_CHECKPOINT to the latest .pth file.


## Initialization

In [1]:
import sys, json, copy
from pathlib import Path

import torch
import timm
import wandb


PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage4_leaf_jepa_pretraining.config_stage4 import *
from stage4_leaf_jepa_pretraining.pretrain_utils import (
    set_seed, get_pretrain_transform, PlantVillagePretrainDataset,
    MultiBlockMasking, DiseaseRegionBiasedMasking, SaliencyMap,
    IJEPAPredictor, EMAUpdater, get_layerwise_optimizer,
    WarmupCosineScheduler, pretrain_one_epoch,
    LinearProbeMonitor, save_checkpoint, load_checkpoint,
    plot_pretrain_curves
)

from stage2_dataset_preparation.outputs.augmentation.transforms import (
    get_pretrain_transform, get_eval_transform, get_finetune_transform
)

set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)
PRETRAIN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
verify_config()

# Optionally resume from checkpoint
RESUME_FROM_CHECKPOINT = None  # e.g. PRETRAIN_CKPT_DIR / "epoch_0050.pth"


✅ Stage 4 config verified.


# Bulid (Encoders & Predictor)

In [2]:
print("Building model components...")

context_encoder = timm.create_model(
    VIT_MODEL_NAME, pretrained=False, num_classes=0, global_pool="", no_embed_class=True
)
ckpt      = torch.load(IJEPA_CHECKPOINT, map_location="cpu")
state_dict= ckpt.get("target_encoder", ckpt.get("encoder", ckpt))
state_dict= {k.replace("module.", ""): v for k, v in state_dict.items()}
context_encoder.load_state_dict(state_dict, strict=False)
print("  Context encoder: Meta I-JEPA weights loaded ✅")

target_encoder = copy.deepcopy(context_encoder)
for p in target_encoder.parameters():
    p.requires_grad = False
print("  Target encoder: deep copy, all params frozen ✅")

predictor = IJEPAPredictor(
    encoder_embed_dim = VIT_EMBED_DIM, pred_embed_dim = PRED_EMBED_DIM,
    num_patches=NUM_PATCHES, num_heads=PRED_NUM_HEADS, depth=PRED_DEPTH,
    mlp_ratio=PRED_MLP_RATIO, dropout=PRED_DROPOUT,
)
print(f"  Predictor: {sum(p.numel() for p in predictor.parameters()):,} params (random init) ✅")

context_encoder = context_encoder.to(device)
target_encoder  = target_encoder.to(device)
predictor       = predictor.to(device)


Building model components...


/tmp/ipykernel_437/2645726020.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt      = torch.load(IJEPA_CHECKPOINT, map_location="cpu")


  Context encoder: Meta I-JEPA weights loaded ✅
  Target encoder: deep copy, all params frozen ✅
  Predictor: 3,881,984 params (random init) ✅


## DataLoader

In [3]:
transform    = get_pretrain_transform()
csv_path     = SPLITS_DIR / "plantvillage_splits.csv"
train_dataset= PlantVillagePretrainDataset(csv_path, transform=transform)
train_loader = torch.utils.data.DataLoader(
    
    train_dataset, batch_size=PT_BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True,
    worker_init_fn=lambda _: set_seed(RANDOM_SEED),
)
print(f"DataLoader ready: {len(train_dataset):,} images | {len(train_loader)} batches/epoch")


  PlantVillagePretrainDataset: 38,013 training images
DataLoader ready: 38,013 images | 296 batches/epoch


## Masking Strategy

In [4]:
ENABLE_BIASED_MASKING = True  #→ novel contribution (main experiment)
# ENABLE_BIASED_MASKING = False → standard I-JEPA masking (ablation baseline)

masking_kwargs = dict(
    image_size=IMAGE_CROP, patch_size=PATCH_SIZE,
    num_target_blocks=PT_NUM_TARGET_BLOCKS,
    context_scale=PT_CONTEXT_SCALE, context_ratio=PT_CONTEXT_RATIO,
    target_scale=PT_TARGET_SCALE,  target_ratio=PT_TARGET_RATIO,
)

if ENABLE_BIASED_MASKING:
    masking_fn = DiseaseRegionBiasedMasking(
        **masking_kwargs, bias_strength=SALIENCY_BIAS_STRENGTH
    )
    saliency_fn_obj = SaliencyMap(
        patch_size=PATCH_SIZE, image_size=IMAGE_CROP,
        healthy_hue_center=HEALTHY_HUE_CENTER, healthy_hue_sigma=HEALTHY_HUE_SIGMA,
    )
    # Wrap saliency to accept tensors
    def saliency_fn(img_tensor):
        return saliency_fn_obj(img_tensor, NORM_MEAN, NORM_STD)
    print("Masking: Disease-region-biased (novel contribution) ✅")
else:
    masking_fn  = MultiBlockMasking(**masking_kwargs)
    saliency_fn = None
    print("Masking: Standard multi-block (ablation mode)")


Masking: Disease-region-biased (novel contribution) ✅


## Defining (Optimizer, Schedular, EMA)

In [5]:
total_steps = PT_EPOCHS * len(train_loader)
optimizer   = get_layerwise_optimizer(
    context_encoder, predictor,
    frozen_layers=FROZEN_LAYERS, low_lr_layers=LOW_LR_LAYERS, std_lr_layers=STD_LR_LAYERS,
    lr_head=PT_LR_HEAD, lr_mid=PT_LR_ENCODER_MID, lr_top=PT_LR_ENCODER_TOP,
    weight_decay=PT_WEIGHT_DECAY,
)
scheduler = WarmupCosineScheduler(optimizer, warmup_epochs=PT_WARMUP_EPOCHS,
                                   total_epochs=PT_EPOCHS)
ema_updater = EMAUpdater(tau_start=EMA_TAU_START, tau_end=EMA_TAU_END,
                          total_steps=total_steps)

print(f"Optimiser:  AdamW layer-wise LR | predictor={PT_LR_HEAD} | top={PT_LR_ENCODER_TOP} | mid={PT_LR_ENCODER_MID}")
print(f"Scheduler:  WarmupCosine | warmup={PT_WARMUP_EPOCHS} epochs | total={PT_EPOCHS} epochs")
print(f"EMA:        τ {EMA_TAU_START} → {EMA_TAU_END} | total_steps={total_steps:,}")
print(f"Loss:       {PT_LOSS}")


Optimiser:  AdamW layer-wise LR | predictor=0.0003 | top=0.0001 | mid=1e-05
Scheduler:  WarmupCosine | warmup=10 epochs | total=125 epochs
EMA:        τ 0.996 → 0.999 | total_steps=37,000
Loss:       smooth_l1


## LP monitor

In [6]:
lp_monitor = LinearProbeMonitor(
    splits_dir     = SPLITS_DIR,
    norm_mean      = NORM_MEAN,
    norm_std       = NORM_STD,
    num_classes    = NUM_CLASSES,
    monitor_epochs = LP_MONITOR_EPOCHS,
    monitor_frac   = LP_MONITOR_FRAC,
    batch_size     = 256,
    num_workers    = NUM_WORKERS,
    device         = device,
)

os.environ["WANDB__SERVICE_WAIT"] = "10"
os.environ["WANDB_DISABLED"] = "true"
try:
    run = wandb.init(
        project  = WANDB_PROJECT,
        entity   = WANDB_ENTITY,
        name     = wandb_pretrain_run_name("biased" if ENABLE_BIASED_MASKING else "standard"),
        config   = {
            "model":           VIT_MODEL_NAME,
            "embed_dim":       VIT_EMBED_DIM,
            "epochs":          PT_EPOCHS,
            "batch_size":      PT_BATCH_SIZE,
            "lr_head":         PT_LR_HEAD,
            "lr_top":          PT_LR_ENCODER_TOP,
            "lr_mid":          PT_LR_ENCODER_MID,
            "warmup_epochs":   PT_WARMUP_EPOCHS,
            "ema_tau_start":   EMA_TAU_START,
            "ema_tau_end":     EMA_TAU_END,
            "biased_masking":  ENABLE_BIASED_MASKING,
            "bias_strength":   SALIENCY_BIAS_STRENGTH,
            "num_targets":     PT_NUM_TARGET_BLOCKS,
            "pred_depth":      PRED_DEPTH,
            "pred_dim":        PRED_EMBED_DIM,
            "loss":            PT_LOSS,
            "dataset":         "PlantVillage-train",
            "n_train":         len(train_dataset),
        },
        reinit=True,
    )
    print("WandB run initialised ✅")

except:
    print("WandB init failed — training without logging")
    run = False



wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: muh-haleef02 (muh-haleef02-inform) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB run initialised ✅


## Training

In [7]:
# Resume from checkpoint (if needed)
start_epoch = 1
history     = []
lp_history  = []
best_lp_f1  = 0.0
best_epoch  = 0

if RESUME_FROM_CHECKPOINT and Path(RESUME_FROM_CHECKPOINT).exists():
    start_epoch, history, lp_history = load_checkpoint(
        RESUME_FROM_CHECKPOINT,
        context_encoder, target_encoder, predictor,
        optimizer=optimizer, ema_updater=ema_updater,
        device=device,
    )
    start_epoch += 1  # continue from next epoch
    if lp_history:
        best_lp_f1 = max(r["lp_val_macro_f1"] for r in lp_history)
        best_epoch = max(lp_history, key=lambda r: r["lp_val_macro_f1"])["pretrain_epoch"]
    print(f"Resuming from epoch {start_epoch}  |  Best LP F1 so far: {best_lp_f1:.4f}")
else:
    print(f"Training from epoch 1 / {PT_EPOCHS}")


Training from epoch 1 / 125


In [8]:
# MAIN TRAINING LOOP
print("="*65)
print(f"  LEAF-JEPA PRETRAINING  |  {PT_EPOCHS} epochs  |  {len(train_dataset):,} images")
print(f"  Masking: {'biased (novel)' if ENABLE_BIASED_MASKING else 'standard (ablation)'}")
print("="*65)

for epoch in range(start_epoch, PT_EPOCHS + 1):
    scheduler.step(epoch - 1)  # update LR

    epoch_metrics = pretrain_one_epoch(
        context_encoder = context_encoder,
        target_encoder  = target_encoder,
        predictor       = predictor,
        loader          = train_loader,
        masking_fn      = masking_fn,
        saliency_fn     = saliency_fn if ENABLE_BIASED_MASKING else None,
        optimizer       = optimizer,
        ema_updater     = ema_updater,
        device          = device,
        epoch           = epoch,
        total_epochs    = PT_EPOCHS,
        use_amp         = USE_AMP,
        accumulate_steps= PT_ACCUMULATE_GRAD,
        loss_type       = PT_LOSS,
        wandb_run       = run,
    )
    history.append(epoch_metrics)

    print(
    f"Epoch {epoch:3d}/{PT_EPOCHS} | "
    f"Loss: {epoch_metrics['loss']:.4f} | "
    f"τ: {epoch_metrics['tau']:.5f} | "
    f"LR: {epoch_metrics['lr']:.2e} | "
    f"{epoch_metrics['epoch_time']:.0f}s")

    # ── Periodic linear probe monitoring ──
    if epoch % LP_MONITOR_INTERVAL == 0 or epoch == PT_EPOCHS:
        lp_f1 = lp_monitor.run(target_encoder, pretrain_epoch=epoch, wandb_run=run)
        lp_history = lp_monitor.history

        if lp_f1 > best_lp_f1:
            best_lp_f1 = lp_f1
            best_epoch = epoch
            save_checkpoint(epoch, context_encoder, target_encoder, predictor,
                             optimizer, ema_updater, history, lp_history,
                             PRETRAIN_CKPT_DIR, tag="best")
            print(f"  ★ New best LP F1: {lp_f1:.4f} — checkpoint saved (best)")

    if epoch in [1, 5, 10, 25]:
        for i, block in enumerate(context_encoder.blocks):
            grads = [p.grad.norm().item() for p in block.parameters()
                     if p.grad is not None]
            if grads and run:
                run.log({f"grads/block_{i:02d}": sum(grads)/len(grads),
                         "epoch": epoch})

    # ── Periodic checkpoint ──
    if epoch % CKPT_SAVE_INTERVAL == 0:
        save_checkpoint(epoch, context_encoder, target_encoder, predictor,
                         optimizer, ema_updater, history, lp_history,
                         PRETRAIN_CKPT_DIR)
    with torch.no_grad():
        dist = sum(
            (p1 - p2).norm().item()
            for p1, p2 in zip(context_encoder.parameters(),
                              target_encoder.parameters())
        )
    if run: run.log({"pretrain/encoder_target_dist": dist, "epoch": epoch})


    # ── Persist history ──
    with open(PRETRAIN_DIR / "pretrain_history.json", "w") as f:
        json.dump(history, f, indent=2)
    with open(PRETRAIN_DIR / "lp_monitor_history.json", "w") as f:
        json.dump(lp_history, f, indent=2)

print(f"\n{'='*65}")
print(f"  Pretraining complete!")
print(f"  Best LP Val Macro-F1: {best_lp_f1:.4f} at epoch {best_epoch}")
print(f"  Run S4_6_checkpoint_export.ipynb to export the Leaf-JEPA encoder.")
if run: run.finish()


  LEAF-JEPA PRETRAINING  |  125 epochs  |  38,013 images
  Masking: biased (novel)


Epoch   1/125: 100%|██████████| 296/296 [02:42<00:00,  1.82it/s, loss=0.1721, lr=3.0e-05, tau=0.99600]

  ✓ Epoch   1/125 | Loss: 0.2300 | τ: 0.99600 | LR: 3.00e-05 | Time: 164s
Epoch   1/125 | Loss: 0.2300 | τ: 0.99600 | LR: 3.00e-05 | 164s



Epoch   2/125: 100%|██████████| 296/296 [02:44<00:00,  1.80it/s, loss=0.1248, lr=6.0e-05, tau=0.99600]

  ✓ Epoch   2/125 | Loss: 0.1335 | τ: 0.99600 | LR: 6.00e-05 | Time: 165s
Epoch   2/125 | Loss: 0.1335 | τ: 0.99600 | LR: 6.00e-05 | 165s


...


Epoch 124/125: 100%|██████████| 296/296 [02:42<00:00,  1.82it/s, loss=0.1010, lr=2.2e-07, tau=0.99900]

  ✓ Epoch 124/125 | Loss: 0.0924 | τ: 0.99900 | LR: 2.24e-07 | Time: 163s
Epoch 124/125 | Loss: 0.0924 | τ: 0.99900 | LR: 2.24e-07 | 163s



Epoch 125/125: 100%|██████████| 296/296 [02:42<00:00,  1.82it/s, loss=0.1003, lr=5.6e-08, tau=0.99900]


  ✓ Epoch 125/125 | Loss: 0.0929 | τ: 0.99900 | LR: 5.60e-08 | Time: 164s
Epoch 125/125 | Loss: 0.0929 | τ: 0.99900 | LR: 5.60e-08 | 164s

  [LP Monitor] Epoch 125 — training linear probe...
  PlantVillagePretrainDataset: 38,013 training images
  PlantVillagePretrainDataset: 38,013 training images
  [LP Monitor] Val Macro-F1: 0.8561  (best so far: 0.8848)
  Checkpoint saved → /workspace/leaf-jepa/stage4_leaf_jepa_pretraining/outputs/checkpoints/stage4/epoch_0125.pth

  Pretraining complete!
  Best LP Val Macro-F1: 0.8848 at epoch 25
  Run S4_6_checkpoint_export.ipynb to export the Leaf-JEPA encoder.


epoch,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇██
lp_monitor/val_macro_f1,█▁▇▇▃
pretrain/encoder_target_dist,▃▄▆██▇▆▇▇▆▆▇▆▆▆█▇▇▅▅▅▅▄▄▄▄▄▄▄▅▃▃▃▂▂▂▂▁▁▁
pretrain/epoch_time_s,█▁▅▅▅▃▄▅▅▃▄▆▆▆▃▂▅▇▃▄▄▃▂▅▆▄▃▂▄▅▂▄▁▂▃▂▁▂▄▄
pretrain/grad_norm,▅▄▇▆▇█▆▆▇▆▆▆▆▆▅▆▅▅▆▄▃▃▃▃▄▃▃▂▃▃▂▂▂▂▁▁▁▁▁▁
pretrain/loss,▆▇█▆▄▃▃▃▃▃▂▂▂▂▂▂▁▂▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
pretrain/lr,▂▄▇████████▇▇▇▇▇▇▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁
pretrain/tau,▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇██████
epoch,125
lp_monitor/val_macro_f1,0.8561
pretrain/encoder_target_dist,1.0231


In [9]:
# ── Cell 9: Quick post-training plots ─────────────────────────────────────────
plot_pretrain_curves(
    history, lp_history,
    save_path=FIGURES_DIR / "S4_pretrain_curves.png",
    title=f"Leaf-JEPA Pretraining ({'biased' if ENABLE_BIASED_MASKING else 'standard'} masking)"
)
print("\n✅ S4_3 complete. Proceed to S4_6_checkpoint_export.ipynb")


  Pretraining curves saved → /workspace/leaf-jepa/stage4_leaf_jepa_pretraining/outputs/figures/S4_pretrain_curves.png

✅ S4_3 complete. Proceed to S4_6_checkpoint_export.ipynb
